# 04 — Feature Engineering, Splitting and Scaling


> **Notebook 4 of 11** — part of the *Heart Disease Detection using Explainable AI* project.
> Run the notebooks **in order**, from 01 to 11. Each one saves its results to disk so the next
> one can pick them up.

---

## 🎯 Goal of this notebook

Three jobs, all of which must happen **before** any model sees the data:

1. **Engineer** better features — create `pulse_pressure` and recode `gender`
2. **Split** into a training set and a sealed test set
3. **Scale** the numbers so every feature speaks at the same volume

**Outputs:** `models/scaler.pkl` and `data/prepared.npz`

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# These notebooks live in notebooks/, so data and models are one level up.
DATA = "../data"
MODELS = "../models"
os.makedirs(MODELS, exist_ok=True)
print("Setup complete.")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv(f"{DATA}/cleaned_data.csv")
print(f"Loaded {len(df):,} clean patients")

## 1. Feature engineering

**Feature engineering** means creating *new, smarter columns* from existing ones. It is often the
single biggest source of improvement in a project, because it is where you show that you
understand the *subject*, not just the code.

We already created two in notebook 02 while cleaning:

* **`age_years`** — the raw file stored age in days. A model works better with human-scale numbers.
* **`bmi`** — weight ÷ height². Height and weight alone mean little; BMI combines them into the
  one number doctors actually use.

Now we add the third, and recode gender.

In [ ]:
# --- pulse pressure = the gap between the two blood pressure numbers ---
df["pulse_pressure"] = df["ap_hi"] - df["ap_lo"]

print("Pulse pressure explained:")
print("  A reading of 170/90 gives a pulse pressure of 80.")
print("  A reading of 120/80 gives a pulse pressure of 40.")
print("  Doctors read a WIDE gap as a sign of stiffened arteries,")
print("  even when neither individual number looks alarming.\n")
print(df[["ap_hi", "ap_lo", "pulse_pressure"]].head())

print("\nDisease rate by pulse pressure:")
for lo, hi in [(0, 40), (40, 50), (50, 60), (60, 200)]:
    sub = df[df.pulse_pressure.between(lo, hi, inclusive="left")]
    if len(sub):
        print(f"  {lo:3d}-{hi:3d} mmHg : {sub.cardio.mean()*100:5.1f}%  ({len(sub):,} patients)")

In [ ]:
# --- recode gender: original 1 = woman, 2 = man  ->  0 = woman, 1 = man ---
df["gender"] = df["gender"].map({1: 0, 2: 1})

print("Gender recoded to 0 = female, 1 = male.")
print("Why bother? 0/1 is the standard way to store a yes/no feature, and it")
print("makes the SHAP and LIME explanations in notebooks 09 and 10 read naturally.")
print(df["gender"].value_counts().sort_index())

## 2. Choose the final feature list

We feed the model **11 features**. Notice what we leave out:

* We drop the original `age` (in days) because `age_years` replaces it.
* We drop `height` and `weight` because `bmi` already contains both.

Feeding in both the raw and the derived versions would be **redundant**, and worse, it would split
the explanation: SHAP would give half the credit to `weight` and half to `bmi`, making both look
less important than they really are.

In [ ]:
FEATURES = ["age_years", "gender", "bmi", "ap_hi", "ap_lo", "pulse_pressure",
            "cholesterol", "gluc", "smoke", "alco", "active"]
TARGET = "cardio"

X = df[FEATURES].copy()   # the questions
y = df[TARGET].copy()     # the answers

print(f"X (inputs) : {X.shape[0]:,} patients x {X.shape[1]} features")
print(f"y (answer) : {y.shape[0]:,} labels")
X.head()

## 3. Split into training and test sets

This is the **single most important rule in machine learning**: the model must be graded on
questions it has never seen.

Imagine a teacher who hands out the exam paper a week early. Everyone scores 100%, and the score
means nothing. So we lock away 20% of the patients in a sealed envelope and do not open it until
notebook 08.

`stratify=y` keeps the same healthy/diseased ratio in both halves, so the exam is fair.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

print(f"Training set : {len(X_train):>6,} patients   (the models learn from these)")
print(f"Test set     : {len(X_test):>6,} patients   (sealed until notebook 08)")
print(f"\nDisease rate in training : {y_train.mean()*100:.2f}%")
print(f"Disease rate in test     : {y_test.mean()*100:.2f}%")
print("The two rates match, so the split is fair.")

## 4. Feature scaling

Look at the numbers we are about to feed in:

* Blood pressure runs from **90 to 200**
* Smoking is only **0 or 1**

Two of our three models — Logistic Regression and especially the SVM — work by measuring
*distances* between numbers. Without scaling, blood pressure would drown out smoking simply
because its numbers are bigger, not because it matters more.

`StandardScaler` rewrites each column to have an average of 0 and a spread of 1. Now every feature
speaks at the same volume.

### ⚠️ The one mistake that would invalidate the whole project

We call `.fit()` on the **training data only**, then `.transform()` on both sets.

If we fitted the scaler on all the data, information from the test set (its averages, its spread)
would leak backwards into training, and our final score would be a lie. This is called **data
leakage**, and examiners look for it specifically.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # LEARN here (training only!)
X_test_scaled  = scaler.transform(X_test)        # only APPLY here

print("Before scaling (first training patient):")
print(X_train.iloc[0].to_string())
print("\nAfter scaling (same patient):")
print(pd.Series(X_train_scaled[0], index=FEATURES).round(3).to_string())
print("\nEvery number now sits roughly between -3 and +3.")
print("0 means 'exactly average'. +2 means 'well above average'.")

In [ ]:
# Prove the scaler did its job
check = pd.DataFrame({
    "Mean after scaling": X_train_scaled.mean(axis=0).round(3),
    "Std after scaling":  X_train_scaled.std(axis=0).round(3)}, index=FEATURES)
print(check)
print("\nEvery mean is ~0 and every standard deviation is ~1. Correct.")

## 5. Save everything for the next notebooks

In [ ]:
joblib.dump(scaler, f"{MODELS}/scaler.pkl")

np.savez_compressed(
    f"{DATA}/prepared.npz",
    X_train=X_train.values, X_test=X_test.values,
    y_train=y_train.values, y_test=y_test.values,
    features=np.array(FEATURES))

print(f"Saved {MODELS}/scaler.pkl")
print(f"Saved {DATA}/prepared.npz")
print("\nNotebooks 05, 06 and 07 will load these and train one model each.")

---
## ✅ What we did

* Engineered **`pulse_pressure`**, and recoded gender to 0/1.
* Chose **11 features**, deliberately dropping redundant raw columns.
* Split **80/20 with stratification**, sealing the test set away.
* Scaled with `StandardScaler` **fitted on training data only** — no leakage.

### ▶️ Next: `05_Logistic_Regression.ipynb` — train the first model.